# TabDPT Classifier — DIMER end-to-end tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/tabdpt-classifier-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/tabdpt-classifier-pipeline/blob/main/tutorials/tabdpt_classifier_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Layer6%2FTabDPT-ffcc4d?style=flat)](https://huggingface.co/Layer6/TabDPT)

**Profile:** `E2E`  
**Notebook specification:** DIMER Notebook Specification `1.0`  
**Repository code revision exercised:** `3f40bb364c2cc72d5e366910172bff92e1d8d5a2`

This notebook demonstrates DIMER tabular classification through the repository's production-facing `TabDPTClassificationPipeline`. TabDPT is an in-context learner: `fit()` fits/records preprocessing state and registers labelled support context; it does **not** gradient-train or fine-tune the TabDPT weights. The upstream package/checkpoint supplies the model; this repository adds DIMER-facing provenance, checksum verification, preprocessing, schema enforcement, evaluation helpers, runtime controls, and the `tabdpt-dimer-context-v3` serving-artifact contract.

**By the end of this notebook you will be able to:** acquire and verify the immutable model; load a public sample or gated bring your own data; validate and condition the pipeline; evaluate with task metrics and a majority-class baseline; run inference on separate new records; export machine-readable outputs plus a DIMER artifact; and verify that serialized artifact across a fresh reconstruction boundary.

This notebook does not demonstrate gradient fine-tuning, regression, causal inference, calibrated deployment thresholds, or production fitness. Public-sample metrics are tutorial/sanity evidence only because upstream pretraining overlap cannot be ruled out. See the repository README, MODEL_CARD.md, TABULAR_CLASSIFICATION_DATASET_SPEC.md, DIMER_CONTRACT.md, upstream TabDPT source, and model repository for authoritative surrounding contracts.


## Prerequisites and runtime contract

Use a fresh Google Colab or compatible Jupyter runtime with Python 3.11–3.13. GPU is recommended; CPU is supported but slower. The demonstrated path forces `use_flash=False` for Tesla T4 portability, uses `compile_model=False`, no notebook-level quantization, and no explicit mixed-precision override. Network access is required once for repository/model acquisition unless cached. Uploaded data stays in the notebook runtime unless you explicitly export it; do not upload restricted data without authorization.

The code revision is an immutable commit and is deliberately retained by repository branch `anchors/notebook-spec-v1-code-20260910`; the branch is a reachability anchor, not the version identifier. The SHA remains the version identifier even if PR #17 is squash- or rebase-merged.

Seeds control documented splitting and model stochasticity, but bitwise identity across devices, CUDA/library builds, and kernels is not promised. DIMER defaults include a 10,000-row support ceiling and supported context range 128–16,384.


In [ ]:
import sys
if "torch" in sys.modules:
    raise RuntimeError("Start from a fresh runtime: install the pinned environment before importing PyTorch.")

REPO_REVISION = "3f40bb364c2cc72d5e366910172bff92e1d8d5a2"
REPO_DIR = "/content/tabdpt-classifier-pipeline"
!rm -rf "$REPO_DIR"
!git clone -q https://github.com/kurtvalcorza/tabdpt-classifier-pipeline.git "$REPO_DIR"
!git -C "$REPO_DIR" checkout -q "$REPO_REVISION"
!python -m pip install -q -r "$REPO_DIR/tutorials/requirements-colab.txt"
!python -m pip install -q --no-deps "$REPO_DIR"


## 1. Verify the runtime and model provenance

This stage reports effective runtime identity and resolves the exact `Layer6/TabDPT` checkpoint at the repository-pinned immutable revision. The `.safetensors` bytes are SHA-256 verified before model construction; model-repository remote Python code is not executed. Success establishes identity and byte integrity, not model quality.


In [ ]:
import importlib.metadata as mdlib
import platform
import torch
from tabdpt_classifier_pipeline import (
    TABDPT_HF_REPO, TABDPT_HF_REVISION, TABDPT_UPSTREAM_CODE_COMMIT,
    TABDPT_WEIGHT_FILENAME, TABDPT_WEIGHT_SHA256, TabDPTClassificationPipeline,
    resolve_tabdpt_weights, validate_dimer_artifact,
)
from tabdpt_classifier_pipeline.dimer_runtime import DimerRuntimeConfig

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
for package in ["tabdpt", "torch", "numpy", "pandas", "scikit-learn", "huggingface-hub", "pyarrow"]:
    print(f"{package}:", mdlib.version(package))
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("CUDA runtime:", torch.version.cuda)
print("Execution assumptions: compile_model=False, use_flash=False, quantization=None, precision=framework/device default")
weights = resolve_tabdpt_weights()
print("Model repository:", TABDPT_HF_REPO)
print("Model revision:", TABDPT_HF_REVISION)
print("Upstream code commit:", TABDPT_UPSTREAM_CODE_COMMIT)
print("Weight file:", TABDPT_WEIGHT_FILENAME)
print("Verified SHA-256:", TABDPT_WEIGHT_SHA256)
print("Resolved local path:", weights)


## 2. Load the default sample or bring your own data

The default path uses the public scikit-learn breast-cancer dataset. `upload_single` accepts one labelled CSV and creates deterministic stratified support/evaluation/new-record partitions. `upload_presplit` preserves `train.csv`, `val.csv`, and unlabelled `new.csv`; use this for temporal, grouped, spatial, patient/device, panel, or other leakage-sensitive data where random row splitting is invalid.

Before upload, the expected schema is one classification target column (default `target`) plus uniquely named feature columns. Support data must contain at least two classes; evaluation classes must occur in support. New data must not contain the target and must match the fitted feature schema.


In [ ]:
DATA_MODE = "sample"  # @param ["sample", "upload_single", "upload_presplit"]
TARGET_COLUMN = "target"  # @param {type:"string"}
DROP_COLUMNS = []
SEED = 42

import csv
from collections import Counter
from pathlib import Path
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

DATA_DIR = Path("/content/tabdpt-tutorial-data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

def read_csv_strict(path):
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        header = next(csv.reader(handle), None)
    if not header:
        raise ValueError(f"{path.name} is empty")
    duplicates = sorted(k for k, v in Counter(header).items() if v > 1)
    if duplicates:
        raise ValueError(f"{path.name} contains duplicate column names: {duplicates}")
    return pd.read_csv(path)

def validate_labeled(frame, name):
    if TARGET_COLUMN not in frame.columns:
        raise ValueError(f"{name} is missing target column {TARGET_COLUMN!r}")
    if frame[TARGET_COLUMN].isna().any():
        raise ValueError(f"{name} target contains missing values")
    if frame.drop(columns=[TARGET_COLUMN, *DROP_COLUMNS], errors="ignore").shape[1] == 0:
        raise ValueError(f"{name} must contain at least one effective feature")

if DATA_MODE == "sample":
    frame = load_breast_cancer(as_frame=True).frame.copy()
    support_eval, new_labeled = train_test_split(frame, test_size=0.10, random_state=SEED, stratify=frame[TARGET_COLUMN])
    support, evaluation = train_test_split(support_eval, test_size=2/9, random_state=SEED, stratify=support_eval[TARGET_COLUMN])
    new_records = new_labeled.drop(columns=[TARGET_COLUMN]).copy()
    DATA_PROVENANCE = "scikit-learn breast cancer dataset; public tutorial sample"
elif DATA_MODE in {"upload_single", "upload_presplit"}:
    from google.colab import files
    uploaded = files.upload()
    for name, payload in uploaded.items():
        (DATA_DIR / Path(name).name).write_bytes(payload)
    if DATA_MODE == "upload_single":
        csv_files = sorted(DATA_DIR.glob("*.csv"))
        if len(csv_files) != 1:
            raise ValueError("upload_single requires exactly one CSV")
        frame = read_csv_strict(csv_files[0]); validate_labeled(frame, csv_files[0].name)
        support_eval, new_labeled = train_test_split(frame, test_size=0.10, random_state=SEED, stratify=frame[TARGET_COLUMN])
        support, evaluation = train_test_split(support_eval, test_size=2/9, random_state=SEED, stratify=support_eval[TARGET_COLUMN])
        new_records = new_labeled.drop(columns=[TARGET_COLUMN]).copy()
        DATA_PROVENANCE = f"user upload: {csv_files[0].name}; deterministic stratified 70/20/10 split"
    else:
        paths = {name: DATA_DIR / name for name in ("train.csv", "val.csv", "new.csv")}
        missing = [name for name, path in paths.items() if not path.exists()]
        if missing:
            raise ValueError(f"upload_presplit missing files: {missing}")
        support = read_csv_strict(paths["train.csv"]); evaluation = read_csv_strict(paths["val.csv"]); new_records = read_csv_strict(paths["new.csv"])
        validate_labeled(support, "train.csv"); validate_labeled(evaluation, "val.csv")
        if TARGET_COLUMN in new_records.columns:
            raise ValueError("new.csv must be unlabelled")
        DATA_PROVENANCE = "user-provided train.csv/val.csv/new.csv partitions preserved"
else:
    raise ValueError(f"Unsupported DATA_MODE: {DATA_MODE!r}")

validate_labeled(support, "support"); validate_labeled(evaluation, "evaluation")
support_classes = set(support[TARGET_COLUMN].map(str)); evaluation_classes = set(evaluation[TARGET_COLUMN].map(str))
if len(support_classes) < 2:
    raise ValueError("Classification requires at least two support classes")
missing_classes = sorted(evaluation_classes - support_classes)
if missing_classes:
    raise ValueError(f"Evaluation classes absent from support: {missing_classes}")
support_hashes = set(pd.util.hash_pandas_object(support, index=False).astype(str))
evaluation_hashes = set(pd.util.hash_pandas_object(evaluation, index=False).astype(str))
if support_hashes & evaluation_hashes:
    raise ValueError("Exact duplicate rows detected across support/evaluation partitions")
if len(support) > 10_000:
    raise ValueError("Support exceeds the DIMER default 10,000-row ceiling; provide an explicit governed subset")
print(DATA_PROVENANCE, support.shape, evaluation.shape, new_records.shape)


## 3. Condition the model on support data

`fit()` uses repository preprocessing and registers support context with the upstream in-context estimator. No model-weight gradient update occurs. Unseen categorical values later use the fitted unknown-category code rather than refitting category maps.


In [ ]:
TUTORIAL_RUNTIME = DimerRuntimeConfig(
    target_column=TARGET_COLUMN, drop_columns=tuple(DROP_COLUMNS), max_train_rows=10000, validation_split=0.2,
    fine_tune=False, n_ensembles=2, context_size=512, batch_size=512, temperature=1.0, seed=SEED,
)
INFERENCE = TUTORIAL_RUNTIME.inference_kwargs()
pipe = TabDPTClassificationPipeline(model_weight_path=weights, compile_model=False, use_flash=False, seed=SEED)
pipe.fit(support, target_column=TARGET_COLUMN, drop_columns=DROP_COLUMNS, seed=SEED)

def report_category_drift(frame, label):
    effective = frame.drop(columns=DROP_COLUMNS, errors="ignore")
    required = list(pipe.feature_encoder.feature_columns)
    missing = [c for c in required if c not in effective.columns]
    extra = [c for c in effective.columns if c not in required]
    if missing or extra:
        raise ValueError(f"{label} feature schema mismatch; missing={missing}, extra={extra}")
    for column, mapping in pipe.feature_encoder.category_maps.items():
        observed = {str(v) for v in effective[column].dropna().tolist()}
        unseen = sorted(observed - set(mapping))
        if unseen:
            print(f"WARNING: {label}.{column} has unseen categories: {unseen[:5]}")
report_category_drift(evaluation.drop(columns=[TARGET_COLUMN]), "evaluation")
report_category_drift(new_records, "new_records")
if len(support) > INFERENCE["context_size"]:
    print(f"Context reduction active: saved support rows={len(support)}, per-ensemble context_size={INFERENCE['context_size']}; seeded balanced subsampling applies.")
else:
    print("Context reduction inactive for current support size.")


## 4. Evaluate and compare a majority-class baseline

The repository reports accuracy, log loss, and binary ROC-AUC where defined. Accuracy measures discrete correctness; log loss penalizes poor true-class scores; ROC-AUC measures ranking discrimination across thresholds. No single metric establishes overall quality. This is a single deterministic holdout with no dispersion estimate and no model selection. `predict_proba()` provides probability-normalized class scores, but calibration is not established; the default decision rule is argmax.


In [ ]:
import json
from sklearn.metrics import accuracy_score
metrics = pipe.evaluate(evaluation, **INFERENCE)
majority_class = support[TARGET_COLUMN].map(str).mode().iloc[0]
baseline_accuracy = float(accuracy_score(evaluation[TARGET_COLUMN].map(str), [majority_class] * len(evaluation)))
evaluation_report = {"estimationProcedure": "single deterministic holdout; no model selection", "tutorialEvidenceOnly": True, "modelMetrics": metrics, "majorityClassBaseline": {"class": majority_class, "accuracy": baseline_accuracy}}
print(json.dumps(evaluation_report, indent=2))


## 5. Run inference on separate new records

These records are separate from evaluation. Output preserves `row_id`, the argmax `prediction`, and one class-score column in exact class order. Scores are not claimed calibrated confidence estimates.


In [ ]:
new_scores = pipe.predict_proba(new_records, **INFERENCE)
new_predictions = pipe.predict(new_records, **INFERENCE)
prediction_table = pd.DataFrame({"row_id": new_records.index.astype(str), "prediction": new_predictions.astype(str)})
for class_name in pipe.class_labels_:
    prediction_table[f"score_{class_name}"] = new_scores[class_name].to_numpy()
print(prediction_table.head())


## 6. Export machine-readable outputs, provenance, and artifact

The deployable serving state is not the checkpoint alone: it also requires labelled support context and fitted preprocessing state. The artifact contains source support data and therefore inherits its confidentiality, licensing, retention, and disclosure obligations.


In [ ]:
import hashlib
from dataclasses import asdict
OUTPUT_DIR = Path("/content/tabdpt-tutorial-output"); ARTIFACT_DIR = OUTPUT_DIR / "artifact"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()
predictions_path = OUTPUT_DIR / "predictions.csv"; metrics_path = OUTPUT_DIR / "metrics.json"; provenance_path = OUTPUT_DIR / "provenance.json"
context_path = ARTIFACT_DIR / "training_context.parquet"; manifest_path = ARTIFACT_DIR / "artifact.json"
prediction_table.to_csv(predictions_path, index=False); metrics_path.write_text(json.dumps(evaluation_report, indent=2) + "\n", encoding="utf-8")
support.to_parquet(context_path, index=False)
preprocessing_state = pipe.export_preprocessing_state()
runtime_config = asdict(TUTORIAL_RUNTIME); runtime_config["drop_columns"] = list(runtime_config["drop_columns"])
manifest = {
    "format": "tabdpt-dimer-context-v3", "taskType": "tabular_classification",
    "targetColumn": TARGET_COLUMN, "dropColumns": list(preprocessing_state["dropColumns"]), "classNames": list(pipe.class_labels_),
    "runtimeConfig": runtime_config, "preprocessing": preprocessing_state,
    "baseModel": {"repo": TABDPT_HF_REPO, "revision": TABDPT_HF_REVISION, "filename": TABDPT_WEIGHT_FILENAME, "sha256": TABDPT_WEIGHT_SHA256, "upstreamCodeCommit": TABDPT_UPSTREAM_CODE_COMMIT},
    "trainingContext": {"path": context_path.name, "sizeBytes": context_path.stat().st_size, "sha256": sha256_file(context_path)},
}
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
provenance = {
    "repository": "kurtvalcorza/tabdpt-classifier-pipeline", "repositoryRevision": REPO_REVISION, "notebookProfile": "E2E", "notebookSpecVersion": "1.0",
    "dataProvenance": DATA_PROVENANCE, "model": manifest["baseModel"], "classOrder": list(pipe.class_labels_), "runtimeConfig": runtime_config,
    "runtime": {"python": sys.version.split()[0], "platform": platform.platform(), "tabdpt": mdlib.version("tabdpt"), "torch": mdlib.version("torch"), "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU", "cuda": torch.version.cuda, "use_flash": False, "compile_model": False, "quantization": None, "precision": "framework/device default"},
    "adaptation": {"type": "preprocessing fit + in-context support conditioning; no gradient training", "supportRows": len(support), "supportSha256": manifest["trainingContext"]["sha256"], "seed": runtime_config["seed"]},
}
provenance_path.write_text(json.dumps(provenance, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(predictions_path, metrics_path, provenance_path, manifest_path, context_path)


## 7. Verify the serialized artifact across a fresh reconstruction boundary

The reload path deliberately discards the producer object and all producer-side runtime-control variables. It copies only serialized artifact files, validates them, reconstructs the serving state, and derives inference controls exclusively from `validated_manifest['runtimeConfig']`. This distinguishes merely loading files from reproducing equivalent outputs.


In [ ]:
import gc, shutil
import numpy as np
probe = new_records.iloc[:min(10, len(new_records))].copy()
before_classes = pipe.predict(probe, **INFERENCE).astype(str).to_numpy()
before_scores = pipe.predict_proba(probe, **INFERENCE).to_numpy()
del pipe, INFERENCE, TUTORIAL_RUNTIME, SEED, runtime_config, weights
gc.collect()
RELOAD_DIR = Path("/content/tabdpt-artifact-reload")
shutil.rmtree(RELOAD_DIR, ignore_errors=True); RELOAD_DIR.mkdir(parents=True)
shutil.copy2(manifest_path, RELOAD_DIR / "artifact.json"); shutil.copy2(context_path, RELOAD_DIR / "training_context.parquet")
validated_manifest, validated_context = validate_dimer_artifact(RELOAD_DIR / "artifact.json", strict_directory=True)
reload_runtime = validated_manifest["runtimeConfig"]
reload_inference = {key: reload_runtime[key] for key in ("n_ensembles", "context_size", "batch_size", "temperature", "seed")}
reloaded = TabDPTClassificationPipeline.load_artifact(RELOAD_DIR / "artifact.json", compile_model=False, use_flash=False, seed=reload_runtime["seed"])
after_classes = reloaded.predict(probe, **reload_inference).astype(str).to_numpy()
after_scores = reloaded.predict_proba(probe, **reload_inference).to_numpy()
if not np.array_equal(before_classes, after_classes):
    raise AssertionError("Serialized artifact reload changed discrete predictions")
if not np.allclose(before_scores, after_scores, rtol=1e-6, atol=1e-7):
    raise AssertionError(f"Serialized artifact reload changed class scores; max_abs={float(np.max(np.abs(before_scores-after_scores)))}")
print("Fresh-boundary validation/reconstruction: PASS", validated_context)
print("Exact class equivalence: PASS; score equivalence rtol=1e-6 atol=1e-7: PASS")


## Interpretation, limits, and next steps

A successful top-to-bottom run proves that the pinned code/model path can validate data, condition the in-context classifier, compute tutorial metrics, score separate records, export a full production-shaped DIMER v3 artifact, validate it, and reconstruct equivalent probe outputs using only serialized serving state plus explicitly permitted base-model acquisition. It does **not** prove accuracy, calibration, fairness, robustness, security, or production fitness for a particular domain. Random splits are inappropriate for leakage-sensitive data, a single holdout has no uncertainty estimate, public-sample overlap with pretraining cannot be ruled out, and seeded execution does not imply cross-hardware bitwise identity.

Before release, record clean-runtime execution evidence for the exact release revision and then run the companion artifact-inference notebook in a separate clean session with the exported artifact and genuinely external new input. Static CI is not execution evidence.
